In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnableParallel, branch
from dataclasses import Field

from  langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel
from typing import Literal, TypedDict
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END
from IPython.display import Image,display, Markdown



# This function will load all the variables from the .env file and will
# make them available in the os.environ dictionary (env variables)
load_dotenv()
ssl_cert_file=""

# Set SSL certificate for requests library (used by tiktoken)
os.environ['REQUESTS_CA_BUNDLE'] = ssl_cert_file
os.environ['SSL_CERT_FILE'] = ssl_cert_file

client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()

from langchain_core.documents import Document

## Your text data
my_text = """Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many AI applications are not perceived as AI: "A lot of cutting edge AI has filtered into general applications, often without being called AI because once something becomes useful enough and common enough it's not labeled AI anymore."[2][3]

Various subfields of AI research are centered around particular goals and the use of particular tools. The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, perception, and support for robotics.[a] To reach these goals, AI researchers have adapted and integrated a wide range of techniques, including search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[4] Some companies, such as OpenAI, Google DeepMind and Meta,[5] aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human.
"""


docs = [Document(page_content=my_text,metadata={"source":"ABC","documentID":"Doc1"})]
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50)

chunks = splitter.split_documents(docs)
chunks

# DSX Embedding Interface (as per documentation)
from langchain_core.embeddings import Embeddings
from typing import List

class DSXEmbeddingInterface(Embeddings):
      
    def __init__(self, url, model_name, http_client, default_headers):
        self.client = OpenAI(
            base_url=url,
            http_client=http_client,
            api_key='not-empty',  # This is replaced with the token generation based on the authentication provider, but cannot be left blank due to a check in the OpenAI client
            default_headers=default_headers
        )
        self.model_name = model_name
  
      
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """Embed search docs."""
        resp = self.client.embeddings.create(model=self.model_name,
                                             input=texts)
        return [a.embedding for a in  resp.data]
          
  
  
    def embed_query(self, text: str) -> List[float]:
        """Embed query text."""
        return self.embed_documents([text])[0]

# Create embedding model with DSX interface
embedding_model = DSXEmbeddingInterface(
    url="",
    model_name="",
    http_client=httpx.Client(verify=ssl.create_default_context(cafile=ssl_cert_file)),
    default_headers={
        "x-correlation-id": str(uuid.uuid4()),
        'accept': '*/*',
        'Content-Type': 'application/json',
        'Authorization': f"Basic {base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode()}"
    }
)

# Pure numpy-based vector store (no external dependencies)
import numpy as np

class SimpleVectorStore:
    def __init__(self, documents, embedding_model):
        self.documents = documents
        self.embedding_model = embedding_model
        self.embeddings = None
        
    def add_documents(self):
        texts = [doc.page_content for doc in self.documents]
        self.embeddings = np.array(self.embedding_model.embed_documents(texts))
        
    def similarity_search(self, query, k=3):
        query_embedding = np.array(self.embedding_model.embed_query(query))
        # Manual cosine similarity calculation
        query_norm = np.linalg.norm(query_embedding)
        doc_norms = np.linalg.norm(self.embeddings, axis=1)
        similarities = np.dot(self.embeddings, query_embedding) / (doc_norms * query_norm)
        top_k_indices = np.argsort(similarities)[::-1][:k]
        return [self.documents[i] for i in top_k_indices]

# Create and use the simple vector store with DSX embeddings
vectorstore = SimpleVectorStore(chunks, embedding_model)
vectorstore.add_documents()
vectorstore.similarity_search("What is AI?", k=3)
context = vectorstore.similarity_search("What is AI?", k= 3)
response = llm_openai.invoke(f"What is AI? You can answer using the following context: {context}")
print(response.content)